In [1]:
import torch

# Check if GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")


🔹 Using device: cuda


In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from collections import Counter

In [3]:
class AttentionLayer(nn.Module):
    """Self-attention layer for feature interactions."""
    def __init__(self, input_dim, attention_dim):
        super(AttentionLayer, self).__init__()
        self.query = nn.Linear(input_dim, attention_dim)
        self.key = nn.Linear(input_dim, attention_dim)
        self.value = nn.Linear(input_dim, attention_dim)
        self.scale = float(attention_dim) ** 0.5  # ✅ Convert to float to avoid device issues

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # ✅ No more device mismatch
        attention_weights = F.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

class RelevanceModule(nn.Module):
    """Relevance prediction using an attention-enhanced neural network."""
    def __init__(self, input_dim, hidden_dim, attention_dim):
        super(RelevanceModule, self).__init__()
        self.attention = AttentionLayer(input_dim, attention_dim)
        self.fc1 = nn.Linear(attention_dim,128 )
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)
    
    def forward(self, x):
        att_output, _ = self.attention(x)
        x = F.relu(self.fc1(att_output))
        x = self.fc2(x)
        x = F.softmax(self.fc3(x), dim=-1)
        return x[:, 0]  # Return P(r=1|x)

class BiasModule(nn.Module):
    def __init__(self, bias_dim, position_dim, hidden_dim):
        super(BiasModule, self).__init__()
        self.fc1 = nn.Linear(bias_dim + position_dim, 64)  # ✅ Adjust input size
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 4)  # for 2×2 transition matrix
        #self.fc2 = nn.Linear(hidden_dim, 4)  # 2×2 transition matrix

    def forward(self, b, pos):
        x = torch.cat((b, pos), dim=-1)  # ✅ Combine bias and position features
        x = F.relu(self.fc1(x))
        #x = self.fc2(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        x = x.view(-1, 2, 2)  # Reshape to transition matrix
        return F.softmax(x, dim=-1)


class IINWithAttention(nn.Module):
    def __init__(self, input_dim, bias_dim, position_dim, hidden_dim, attention_dim):
        super(IINWithAttention, self).__init__()
        self.relevance_module = RelevanceModule(input_dim, hidden_dim, attention_dim)
        self.bias_module = BiasModule(bias_dim, position_dim, hidden_dim)  # ✅ Now it accepts 3 arguments

    def forward(self, x, b, pos):
        r = self.relevance_module(x)  # P(r=1|x)
        t = self.bias_module(b, pos)  # Bias transition matrix
    
        y_pred = t[:, 0, 0] * (1 - r) + t[:, 0, 1] * r  # Compute final CTR prediction
        return y_pred.unsqueeze(1)  # ✅ Fix: Reshape to (batch_size, 1)



In [4]:

input_dim = 136  # Features
hidden_dim = 32  # Hidden size
attention_dim = 32 
bias_dim = 1
position_dim=1
# Move model to GPU
model = IINWithAttention(input_dim, bias_dim, position_dim, hidden_dim, attention_dim).to(device)  # ✅ Corrected!




In [5]:
class CTRDataset(Dataset):
    """Dataset loader for MSLR-WEB30K with click simulation and re-sampling (based on the paper)."""
    
    def __init__(self, file_path, scenario=4, device="cpu", K=15, ranked_docs=None):
        """
        Parameters:
            file_path (str): Path to the dataset.
            scenario (int): Click simulation scenario (1-5).
            device (str): CPU or GPU.
            K (int): Additional documents per query (default: 15).
        """
        self.device = device
        self.scenario = scenario
        self.K = K  # Number of extra documents per query
        self.ranked_docs = ranked_docs      
        
        
        # ✅ Load & Resample Data Based on Click Simulation
        self.data, self.qids = self.load_data(file_path)
        
        self.features = torch.tensor(self.data[:, 2:], dtype=torch.float32).to(self.device)
        self.position = torch.tensor(self.data[:, 0], dtype=torch.float32).unsqueeze(1).to(self.device)
        self.bias_features = torch.tensor(self.data[:, 1], dtype=torch.float32).unsqueeze(1).to(self.device)
        feature_index = 13  # title length
        theta_values = self.data[:, 2 + feature_index]
        theta_min = np.min(theta_values)
        theta_max = np.max(theta_values)
        positions = self.data[:, 0]
        unique_positions = np.unique(positions)
        print("🔍 Unique Positions:", unique_positions)

        
         # ✅ Fix the function call (Remove `scenario=` part)
        simulated_clicks = [
    CTRDataset.simulate_clicks(p, r, theta, self.scenario, theta_min, theta_max)
    for r, p, theta in zip(self.data[:, 1], self.data[:, 0], theta_values)
]


        self.labels = torch.tensor(simulated_clicks, dtype=torch.float32).unsqueeze(1).to(self.device)
        self.qids = torch.tensor(self.qids, dtype=torch.int64).to(self.device)  # ✅

    def load_data(self, file_path):
        data = []
        qid_list = []  # <-- Track query IDs
        query_groups = {}

        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                label = int(parts[0])
                query_id = int(parts[1].split(":")[1])
                position = len(query_groups.get(query_id, [])) + 1

                features = np.zeros(136)
                for item in parts[2:]:
                    index, value = item.split(":")
                    features[int(index) - 1] = float(value)

                if query_id not in query_groups:
                    query_groups[query_id] = []
                query_groups[query_id].append([position, label] + list(features))

        # ✅ Add this just before the for-loop over query groups
        if self.ranked_docs:
            print("✅ Using production ranker to reorder documents per query.")

        
        for qid, docs in query_groups.items():
            if self.ranked_docs and qid in self.ranked_docs:
                #print(f"✅ Using production ranker to reorder query {qid}")
                ranked_indices = self.ranked_docs[qid]
                docs = [docs[i] for i in ranked_indices]
            # else: keep original order

            # Sample top 5 + K others
            if len(docs) <= 5:
                sampled_docs = docs
            else:
                top_5 = docs[:5]
                extra_docs = random.sample(docs[5:], min(self.K, len(docs) - 5))
                sampled_docs = top_5 + extra_docs

            # ✅ Reassign positions to avoid 0.0 issues
            for new_pos, doc in enumerate(sampled_docs, start=1):
                doc[0] = float(new_pos)

            data.extend(sampled_docs)
            qid_list.extend([qid] * len(sampled_docs))

        return np.array(data), np.array(qid_list)

    @staticmethod
    def linear_observation_weight(theta, theta_min, theta_max):
        """
        Linearly transforms theta to the range [0.5, 1].
        """
        normalized = (theta - theta_min) / (theta_max - theta_min + 1e-8)  # Prevent div by zero
        return 0.5 + normalized


    def simulate_clicks(position, relevance, theta, scenario, theta_min, theta_max):
        """
        Simulates clicks based on position, relevance, and click simulation scenario.
    
        Parameters:
        relevance (int): Document relevance score (0-4).
        position (int): Document position in search results (1-based index).
        scenario (int): Click simulation scenario (1-5).
        theta (float): Parameter for scenario S5 (document effect).
        title_length (float): Title length feature (only for S5).
        
        Returns:
            int: Simulated click (1 if clicked, 0 otherwise).
        """
    # Step 1: Compute Observation Probability (P(o_d = 1 | pos_d))
        if position <= 0:
            raise ValueError(f"Invalid position: {position}")

        
        if scenario <= 4:  # S1-S4: Observation depends only on position
            position = max(position, 1)  # Ensure position is at least 1
            observation_prob = 1 / position if position <= 5 else 0.1
        else:  # Scenario 5
            omega_theta = CTRDataset.linear_observation_weight(theta, theta_min, theta_max)
            position = max(position, 1)  # Prevent divide-by-zero
            observation_prob = omega_theta / position if position <= 5 else 0.1 * omega_theta

        observed = np.random.rand() < observation_prob 

        # Step 2: Compute Click Probability (P(c_d = 1 | pos_d, r_d, o_d))
        if scenario == 1:
            click_prob = 1 if relevance == 1 and observed else 0

        elif scenario == 2:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else 0

        elif scenario == 3:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0)

        elif scenario == 4:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0.01)

        elif scenario == 5:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0)

        # Generate Click
        click = np.random.rand() < click_prob
        return int(click)

    

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Returns (features, bias, position, label)"""
        return self.features[idx], self.bias_features[idx], self.position[idx], self.labels[idx],self.qids[idx]  # ✅ Return qid

In [6]:
def train_production_ranker(dataset, train_ratio=0.1, max_attempts=50):
    """
    Train a simple production ranker using a small subset of training data.
    
    Ensures at least one `1` and one `0` are always included.
    """
    dataloader = DataLoader(dataset, batch_size=len(dataset))  # Load all data at once
    features, _, _, relevance, _ = next(iter(dataloader))


    features = features.cpu().numpy()
    relevance = relevance.cpu().numpy().flatten()

    # ✅ Ensure dataset has both `1` and `0` labels
    ones = [i for i, r in enumerate(relevance) if r == 1]
    zeros = [i for i, r in enumerate(relevance) if r == 0]

    if len(ones) == 0 or len(zeros) == 0:
        print("❌ No positive or negative samples found in dataset!")
        return None, None

    for attempt in range(max_attempts):
        sample_size = max(100, int(len(features) * train_ratio))  # Ensure at least 100 samples
        sample_size = min(sample_size, len(features))  # Avoid exceeding dataset size

        # ✅ Always include at least one `1` and one `0`
        num_ones = min(len(ones), sample_size // 2)
        num_zeros = min(len(zeros), sample_size - num_ones)

        sampled_indices = random.sample(ones, num_ones) + random.sample(zeros, num_zeros)
        train_features = features[sampled_indices]
        train_labels = np.array([1 if r >= 1 else 0 for r in relevance[sampled_indices]])

        label_counts = Counter(train_labels)
        if len(label_counts) > 1:
            break  # ✅ Valid dataset found
        print(f"⚠️ Re-sampling attempt {attempt + 1}: Only one class found, increasing sample size...")

    else:
        print("❌ Failed to find a balanced sample after multiple attempts. Returning None.")
        return None, None

    # Scale features
    scaler = MinMaxScaler()
    train_features = scaler.fit_transform(train_features)

    # Train logistic regression model
    model = LogisticRegression(random_state=42, class_weight='balanced', solver='saga', max_iter=500)
    model.fit(train_features, train_labels)

    print(f"✅ Production Ranker Trained on {len(train_features)} Samples")
    return model, scaler


In [7]:
from collections import defaultdict
import numpy as np

def rank_documents(model, scaler, dataset):
    """
    Rank documents per query using the trained production ranker.
    Returns a dictionary: {qid: [doc_indices_sorted_by_score]}
    """
    dataloader = DataLoader(dataset, batch_size=len(dataset))
    features, _, _, relevance, qid = next(iter(dataloader))

    # Convert tensors to numpy
    features = features.cpu().numpy()
    qid = qid.cpu().numpy()

    # Scale features
    features_scaled = scaler.transform(features)

    # Predict relevance scores
    scores = model.predict_proba(features_scaled)[:, 1]

    # Group document indices by qid
    qid_to_indices = defaultdict(list)
    for idx, q in enumerate(qid):
        qid_to_indices[q].append(idx)

    # Rank within each query
    ranked_docs = {}
    for q, doc_indices in qid_to_indices.items():
        # Pair each document's local index (0 to len-1) with its score
        local_doc_scores = [(i, scores[doc_indices[i]]) for i in range(len(doc_indices))]
        local_doc_scores.sort(key=lambda x: -x[1])  # Sort by score descending
        ranked_docs[q] = [i for i, _ in local_doc_scores]  # local indices

    """ for q, doc_indices in qid_to_indices.items():
        doc_scores = [(idx, scores[idx]) for idx in doc_indices]
        doc_scores.sort(key=lambda x: -x[1])  # Sort by score descending
        ranked_docs[q] = [idx for idx, _ in doc_scores]"""

    return ranked_docs


In [8]:
def train_model(model, dataloader, optimizer, epochs=70, device="cpu"):
    """
    Train the model with click simulation.
    
    Parameters:
        model: The neural network model.
        dataloader: The DataLoader for training data.
        optimizer: The optimizer for training.
        epochs: Number of epochs.
        device: The device (CPU or GPU).
    """
    criterion = nn.BCELoss()
    model.to(device)  # Move model to GPU
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0
        for x, b, pos, y, _ in dataloader:  # ✅ discard qid

            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)  # Move batch to GPU
            
            optimizer.zero_grad()
            y_pred = model(x, b, pos)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(dataloader):.4f}")


In [9]:
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict
import torch

def dcg_at_k(r, k):
    """
    Calculate Discounted Cumulative Gain at rank k
    Args:
        r: Relevance scores (list/array)
        k: Rank cutoff
    Returns:
        DCG@k value
    """
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k):
    """
    Calculate Normalized Discounted Cumulative Gain at rank k
    Args:
        r: Relevance scores (list/array)  
        k: Rank cutoff
    Returns:
        NDCG@k value
    """
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.0
    return dcg_at_k(r, k) / dcg_max

def precision_at_k(r, k):
    """
    Calculate Precision at rank k
    Args:
        r: Relevance scores (binary: 0 or 1)
        k: Rank cutoff
    Returns:
        Precision@k value
    """
    assert k >= 1
    r = np.asarray(r)[:k] != 0
    if r.size != k:
        raise ValueError('Relevance score length < k')
    return np.mean(r)

def average_precision(r):
    """
    Calculate Average Precision
    Args:
        r: Relevance scores (binary: 0 or 1)
    Returns:
        Average Precision value
    """
    r = np.asarray(r) != 0
    out = [precision_at_k(r, k + 1) for k, rel in enumerate(r) if rel]
    if not out:
        return 0.0
    return np.mean(out)

def reciprocal_rank(r):
    """
    Calculate Reciprocal Rank
    Args:
        r: Relevance scores (binary: 0 or 1)
    Returns:
        Reciprocal Rank value
    """
    r = np.asarray(r) != 0
    ranks = np.where(r)[0]
    if len(ranks) > 0:
        return 1.0 / (ranks[0] + 1)
    return 0.0

def evaluate_comprehensive_metrics(model, dataloader, device="cpu"):
    """
    Comprehensive evaluation with multiple ranking metrics
    """
    model.to(device)
    model.eval()
    
    # Store predictions and labels per query
    query_predictions = defaultdict(list)
    query_labels = defaultdict(list)
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for x, b, pos, y, qids in dataloader:
            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
            y_pred = model(x, b, pos)
            
            # Convert to CPU for processing
            predictions = y_pred.cpu().numpy().flatten()
            labels = y.cpu().numpy().flatten() 
            qids_np = qids.cpu().numpy()
            
            # Store for AUC calculation
            all_preds.extend(predictions)
            all_labels.extend(labels)
            
            # Group by query ID
            for pred, label, qid in zip(predictions, labels, qids_np):
                query_predictions[qid].append(pred)
                query_labels[qid].append(label)
    
    # Calculate AUC
    try:
        auc = roc_auc_score(all_labels, all_preds)
    except ValueError as e:
        print(f"⚠️ AUC calculation failed: {e}")
        auc = 0.0
    
    # Calculate ranking metrics per query, then average
    ndcg_1_scores = []
    ndcg_3_scores = []
    ndcg_5_scores = []
    ndcg_10_scores = []
    map_scores = []
    mrr_scores = []
    p_1_scores = []
    p_3_scores = []
    p_5_scores = []
    p_10_scores = []
    
    for qid in query_predictions:
        # Get predictions and labels for this query
        preds = np.array(query_predictions[qid])
        labels = np.array(query_labels[qid])
        
        # Skip queries with no relevant documents
        if np.sum(labels) == 0:
            continue
            
        # Sort documents by prediction score (descending)
        sorted_indices = np.argsort(preds)[::-1]
        sorted_labels = labels[sorted_indices]
        
        # Convert labels to binary relevance (assuming labels are already 0/1 from click simulation)
        # For MSLR-WEB30K original labels (0-4), you might want: binary_labels = (sorted_labels >= 2).astype(int)
        binary_labels = sorted_labels.astype(int)
        
        # Calculate NDCG@k (using original labels for graded relevance)
        if len(sorted_labels) >= 1:
            ndcg_1_scores.append(ndcg_at_k(sorted_labels, 1))
        if len(sorted_labels) >= 3:
            ndcg_3_scores.append(ndcg_at_k(sorted_labels, 3))
        if len(sorted_labels) >= 5:
            ndcg_5_scores.append(ndcg_at_k(sorted_labels, 5))
        if len(sorted_labels) >= 10:
            ndcg_10_scores.append(ndcg_at_k(sorted_labels, 10))
        
        # Calculate MAP and MRR
        map_scores.append(average_precision(binary_labels))
        mrr_scores.append(reciprocal_rank(binary_labels))
        
        # Calculate Precision@k
        if len(binary_labels) >= 1:
            p_1_scores.append(precision_at_k(binary_labels, 1))
        if len(binary_labels) >= 3:
            p_3_scores.append(precision_at_k(binary_labels, 3))
        if len(binary_labels) >= 5:
            p_5_scores.append(precision_at_k(binary_labels, 5))
        if len(binary_labels) >= 10:
            p_10_scores.append(precision_at_k(binary_labels, 10))
    
    # Calculate mean scores
    results = {
        'AUC': auc,
        'NDCG@1': np.mean(ndcg_1_scores) if ndcg_1_scores else 0.0,
        'NDCG@3': np.mean(ndcg_3_scores) if ndcg_3_scores else 0.0,
        'NDCG@5': np.mean(ndcg_5_scores) if ndcg_5_scores else 0.0,
        'NDCG@10': np.mean(ndcg_10_scores) if ndcg_10_scores else 0.0,
        'MAP': np.mean(map_scores) if map_scores else 0.0,
        'MRR': np.mean(mrr_scores) if mrr_scores else 0.0,
        'P@1': np.mean(p_1_scores) if p_1_scores else 0.0,
        'P@3': np.mean(p_3_scores) if p_3_scores else 0.0,
        'P@5': np.mean(p_5_scores) if p_5_scores else 0.0,
        'P@10': np.mean(p_10_scores) if p_10_scores else 0.0,
    }
    
    # Print results
    print("\n" + "="*50)
    print("📊 COMPREHENSIVE RANKING EVALUATION RESULTS")
    print("="*50)
    print(f"🎯 AUC Score:        {results['AUC']:.4f}")
    print(f"📈 NDCG@1:          {results['NDCG@1']:.4f}")
    print(f"📈 NDCG@3:          {results['NDCG@3']:.4f}")
    print(f"📈 NDCG@5:          {results['NDCG@5']:.4f}")
    print(f"📈 NDCG@10:         {results['NDCG@10']:.4f}")
    print(f"🗺️  MAP:             {results['MAP']:.4f}")
    print(f"🥇 MRR:             {results['MRR']:.4f}")
    print(f"🎯 Precision@1:     {results['P@1']:.4f}")
    print(f"🎯 Precision@3:     {results['P@3']:.4f}")
    print(f"🎯 Precision@5:     {results['P@5']:.4f}")
    print(f"🎯 Precision@10:    {results['P@10']:.4f}")
    print(f"📊 Evaluated on {len(query_predictions)} queries")
    print("="*50)
    
    return results

# Replace your existing evaluate_model function call with this:
def evaluate_model_legacy(model, dataloader, device="cpu"):
    """
    Legacy evaluation function (keeping for backward compatibility)
    """
    model.to(device)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, b, pos, y, _ in dataloader:
            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
            y_pred = model(x, b, pos)
            all_preds.extend(y_pred.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    auc = roc_auc_score(all_labels, all_preds)
    print(f"🎯 AUC Score: {auc:.4f}")
    return auc

In [10]:
def evaluate_model(model, dataloader, device="cpu"):
    """
    Evaluate the model on test data.
    """
    model.to(device)  # Move model to GPU
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, b, pos, y, _ in dataloader:  # ✅ now matches 5 outputs

            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)  # Move to GPU

            y_pred = model(x, b, pos)
            all_preds.extend(y_pred.cpu().numpy())  # Move back to CPU for numpy processing
            all_labels.extend(y.cpu().numpy())  # Move back to CPU for numpy processing

    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(all_labels, all_preds)
    print(f"🎯 AUC Score: {auc:.4f}")
    

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [12]:
from sklearn.metrics import ndcg_score
import numpy as np

def evaluate_ndcg_k_range_simple(model, dataloader, device="cpu", max_k=10):
    """
    Evaluate NDCG@k for k = 1 to max_k, assuming each batch corresponds to one query.
    
    Parameters:
        model: Trained model
        dataloader: Yields batches (x, b, pos, y) — no query ID required
        device: 'cuda' or 'cpu'
        max_k: Max value of k to compute NDCG@k
    """
    model.to(device)
    model.eval()
    
    ndcg_sums = np.zeros(max_k)
    batch_count = 0

    with torch.no_grad():
        for x, b, pos, y in dataloader:
            x, b, pos = x.to(device), b.to(device), pos.to(device)
            y_pred = model(x, b, pos).view(-1).cpu().numpy()
            y_true = y.view(-1).cpu().numpy()

            # Skip if batch has fewer than 2 documents
            if len(y_true) < 2:
                continue

            for k in range(1, max_k + 1):
                score = ndcg_score([y_true], [y_pred], k=k)
                ndcg_sums[k - 1] += score

            batch_count += 1

    if batch_count == 0:
        print("⚠️ No valid batches (must have ≥2 items)")
        return

    avg_ndcgs = ndcg_sums / batch_count
    print(f"📊 NDCG@k from 1 to {max_k}")
    for k, score in enumerate(avg_ndcgs, start=1):
        print(f"  NDCG@{k}: {score:.4f}")


In [13]:
def sample_mslr_test(file_path, sample_size=1000):
    """Creates a smaller test dataset for quick evaluation."""
    sampled_lines = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= sample_size:
                break
            sampled_lines.append(line)

    test_file = "sample_mslr_test.txt"
    with open(test_file, 'w') as f:
        f.writelines(sampled_lines)

    print(f"✅ Created test dataset: {test_file} ({sample_size} samples)")
    return test_file

# Run sampling function
test_file_path = "C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\test.txt"  # Original test dataset
sample_test_file = sample_mslr_test(test_file_path, sample_size=1000)

✅ Created test dataset: sample_mslr_test.txt (1000 samples)


In [14]:
file_path="C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\train.txt"
def sample_mslr(file_path, sample_size=5000):
    """Reads only the first N lines from MSLR dataset."""
    sampled_lines = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= sample_size:
                break
            sampled_lines.append(line)
    
    sample_file = "sample_mslr_train_fold1.txt"
    with open(sample_file, 'w') as f:
        f.writelines(sampled_lines)
    
    print(f"Saved {sample_size} lines to {sample_file}")
    return sample_file

# Create a small dataset
sample_file = sample_mslr(file_path, sample_size=5000)

Saved 5000 lines to sample_mslr_train_fold1.txt


In [15]:
# ✅ Step 1: Load the Full Training Dataset (Real Labels)
train_file = "C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\train.txt"
train_dataset_full = CTRDataset(train_file, scenario=4, device=device)

# ✅ Check label distribution before training
dataloader = DataLoader(train_dataset_full, batch_size=len(train_dataset_full))  # Load all data at once
features, bias, position, relevance, qid = next(iter(dataloader))

full_labels = relevance.flatten().cpu().numpy()

label_counts = Counter(full_labels)

print(f"🔍 Label distribution in full training set: {label_counts}")

# ✅ Step 2: Train the Production Ranker on a 1% Subset
print("🔹 Training Production Ranker...")
production_ranker, scaler = train_production_ranker(train_dataset_full, train_ratio=0.05)

# ✅ Step 3: Check if Ranker & Scaler Are Valid
if production_ranker is None or scaler is None:
    print("❌ Production Ranker Training Failed! Using Default Labels Instead.")
    use_ranker = False  # Disable ranker-based ranking
else:
    print("✅ Production Ranker Successfully Trained.")
    use_ranker = True  # Enable ranker-based ranking

# ✅ Step 4: Generate Initial Document Rankings (`R̂_q`) Only If Ranker Exists
if use_ranker:
    print("🔹 Ranking Documents Using Production Ranker...")
    ranked_docs = rank_documents(production_ranker, scaler, train_dataset_full)
else:
    print("none")
    ranked_docs = None  # Skip ranking if ranker failed


# ✅ Step 5: Train the Main Model Using Simulated Clicks
print("🔹 Simulating Clicks and Training Model...")

# Load Training Data with Click Simulation (Using Ranked Documents)
train_dataset = CTRDataset(train_file, scenario=4, device=device, ranked_docs=ranked_docs)


train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)



# Train the Model with Simulated Click Labels
train_model(model, train_dataloader, optimizer, epochs=70, device=device)

# ✅ Step 6: Load Test Data (Real Labels: Relevance ≥ 2 → 1)
test_file = "C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\test.txt"
test_dataset = CTRDataset(test_file, scenario=4, device=device)  # Test data should NOT be simulated!
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# ✅ Step 7: Evaluate Model on Real Test Data
evaluate_model_legacy(model, test_dataloader, device=device)
# After training
#evaluate_ndcg_k_range(model, test_dataloader, device=device)
# ✅ Step 7: Comprehensive Evaluation on Real Test Data
print("🔹 Starting Comprehensive Evaluation...")
results = evaluate_comprehensive_metrics(model, test_dataloader, device=device)

# Optional: Save results to file for paper
import json
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✅ Results saved to 'evaluation_results.json'")




🔍 Unique Positions: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20.]
🔍 Label distribution in full training set: Counter({np.float32(0.0): 328771, np.float32(1.0): 41033})
🔹 Training Production Ranker...
✅ Production Ranker Trained on 18490 Samples
✅ Production Ranker Successfully Trained.
🔹 Ranking Documents Using Production Ranker...
🔹 Simulating Clicks and Training Model...
✅ Using production ranker to reorder documents per query.
🔍 Unique Positions: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20.]
Epoch 1/70, Loss: 0.2680
Epoch 2/70, Loss: 0.2587
Epoch 3/70, Loss: 0.2578
Epoch 4/70, Loss: 0.2571
Epoch 5/70, Loss: 0.2568
Epoch 6/70, Loss: 0.2565
Epoch 7/70, Loss: 0.2562
Epoch 8/70, Loss: 0.2561
Epoch 9/70, Loss: 0.2560
Epoch 10/70, Loss: 0.2559
Epoch 11/70, Loss: 0.2559
Epoch 12/70, Loss: 0.2559
Epoch 13/70, Loss: 0.2558
Epoch 14/70, Loss: 0.2558
Epoch 15/70, Loss: 0.2558
Epoch 16/70, Loss: 0.2557
Epoch 17/70, Loss: 